---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

*This notebook is the assignment template. Each group works on its own copy, and cells or sections may be added as needed.*

---

## Group 11

- 13524027 Aufa Rienaldifaza Ahmad
- 13524045 Ahmad Zaky Robbani
- 13524075 Hakam Avicena Mustain
- 13524087 Muhammad Fakhry Zaki
- 13524099 Muhammad Akmal

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
# %pip install -q numpy pandas
%pip install gdown imbalanced-learn matplotlib numpy pandas scikit-learn

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
import sys
from pathlib import Path
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import calendar

from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterGrid
from sklearn.feature_selection import mutual_info_classif
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sys.path.insert(0, str(Path.cwd() / 'src'))

pd.set_option('display.max_columns', None)

## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED       = 42
    N_FOLDS = 5 
    DATA_DIR   = Path('data')
    OUTPUT_DIR = Path('output')
    ID_COLUMN = 'id'
    TARGET = 'is_canceled'

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
num_cols = [
    "num_adults", "num_children", "weekend_nights", "weekday_nights",
    "days_before_arrival", "arrival_year", "arrival_month", "arrival_day",
    "prior_cancellations", "prior_completed_bookings",
    "nightly_rate_idr", "special_requests"
]
cat_cols = ["meal_package", "room_category", "booking_channel"]
bin_cols = ["needs_parking", "is_returning_guest"]
target_col = "is_canceled"

print('\nTRAIN INFO')
train_df.info()
print('\nTEST INFO')
test_df.info()

train_df.head()

## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
print(train_df[target_col].value_counts())
print('\nTarget distribution (%)')
print((train_df[target_col].value_counts(normalize=True) * 100).round(2))

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
for axis, column in zip(axes.flat, num_cols):
    axis.hist(train_df[column].dropna(), bins=30)
    axis.set_title(column)
plt.tight_layout()
plt.show()

for column in cat_cols + bin_cols:
    print(f'--- {column} ---')
    print(train_df[column].value_counts(dropna=False))

fig, axes = plt.subplots(1, len(cat_cols), figsize=(15, 5))
for axis, column in zip(axes, cat_cols):
    train_df[column].value_counts(dropna=False).plot(kind='bar', ax=axis)
    axis.set_title(column)
    axis.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

correlation = train_df[num_cols + [target_col]].corr()
print('\nNumeric correlations with target:')
print(correlation[target_col].drop(target_col).sort_values(ascending=False))
fig, axis = plt.subplots(figsize=(10, 8))
image = axis.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
axis.set_xticks(range(len(correlation.columns)), correlation.columns, rotation=45, ha='right')
axis.set_yticks(range(len(correlation.index)), correlation.index)
fig.colorbar(image, ax=axis)
plt.tight_layout()
plt.show()

for column in cat_cols:
    rate = train_df.groupby(column, dropna=False)[target_col].mean().sort_values(ascending=False)
    print(f'\nCancellation rate by {column}:')
    print(rate.round(3))

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
def missing_report(df, name):
    miss = df.isna().sum()
    pct = (miss / len(df) * 100).round(2)
    report = pd.DataFrame({"missing_count": miss, "missing_pct": pct})
    report = report[report["missing_count"] > 0].sort_values("missing_pct", ascending=False)
    print(f"--- Missing values: {name} ---")
    print(report if not report.empty else "Tidak ada missing value.")
    return report

In [ ]:
def iqr_outlier_count(series: pd.Series) -> int:
    values = series.dropna()
    if values.empty:
        return 0

    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    iqr = q3 - q1
    lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((values < lower_bound) | (values > upper_bound)).sum())

In [ ]:
def invalid_date_mask(df: pd.DataFrame) -> pd.Series:
    required_columns = {"arrival_year", "arrival_month", "arrival_day"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(
            f"Missing required columns: {', '.join(sorted(missing_columns))}"
        )

    def is_invalid_date(row: pd.Series) -> bool:
        try:
            max_day = calendar.monthrange(
                int(row["arrival_year"]), int(row["arrival_month"])
            )[1]
            return row["arrival_day"] > max_day
        except (OverflowError, TypeError, ValueError):
            return False

    return df.apply(is_invalid_date, axis=1)

In [ ]:
missing_report(train_df, 'train')
missing_report(test_df, 'test')

print('\nDuplicate rows (train):', train_df.duplicated().sum())
print('Duplicate rows (test):', test_df.duplicated().sum())
print('Duplicate ids (train):', train_df['id'].duplicated().sum())
print('Duplicate ids (test):', test_df['id'].duplicated().sum())

for column in num_cols:
    print(f'{column}: {iqr_outlier_count(train_df[column])} potential outliers (IQR rule)')

zero_nights = train_df[
    (train_df['weekend_nights'] == 0) & (train_df['weekday_nights'] == 0)
]
zero_guests = train_df[
    (train_df['num_adults'] == 0) & (train_df['num_children'] == 0)
]
print('\nBookings with zero nights:', len(zero_nights))
print('Bookings with zero guests:', len(zero_guests))

invalid_dates = train_df.loc[invalid_date_mask(train_df)]
print('Invalid arrival dates:', len(invalid_dates))

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
X_test = test_df.drop(columns=[CFG.ID_COLUMN, CFG.TARGET], errors='ignore').copy()
X = train_df.drop(columns=[CFG.ID_COLUMN, CFG.TARGET])
y = train_df[CFG.TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, train_size=0.75, stratify=y, random_state=CFG.SEED
)
# Preserve raw rows: preprocessing is fitted separately within every CV fold.
X_train_raw, X_val_raw = X_train.copy(), X_val.copy()
pd.Series({'X_train': len(X_train), 'X_val': len(X_val), 'X_test': len(X_test)})

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
# Write your code here

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
class DataCleaner(BaseEstimator, TransformerMixin):
    """Fit imputation on training rows without dropping or reordering samples."""
    def fit(self, X, y=None):
        self.fill_values_ = {}
        for column in X.columns:
            values = X[column]
            if pd.api.types.is_numeric_dtype(values):
                fill = values.median()
            else:
                modes = values.mode()
                fill = modes.iloc[0] if len(modes) else 'Unknown'
            self.fill_values_[column] = 0 if pd.isna(fill) else fill
        return self

    def transform(self, X):
        return X.copy().fillna(self.fill_values_)

In [ ]:
# Cleaned copies are for inspection; modeling keeps raw data until each fold.
cleaner = DataCleaner().fit(X_train_raw)
train_clean = cleaner.transform(X_train_raw)
val_clean = cleaner.transform(X_val_raw)
test_clean = cleaner.transform(X_test)
print('Missing values after cleaning:', {
    name: int(frame.isna().sum().sum())
    for name, frame in [('train', train_clean), ('validation', val_clean), ('test', test_clean)]
})

### Duplicates

This part covers repeated records.

In [ ]:
print('Duplicate rows after cleaning (train):', X_train.duplicated().sum())
print('Duplicate rows after cleaning (validation):', X_val.duplicated().sum())
print('Duplicate rows after cleaning (test):', X_test.duplicated().sum())

### Outliers

This part covers values far from the rest of the data.

In [ ]:
# Inspect extremes without removing genuine booking behavior.
# In particular, do not cap special_requests or booking history counts.
print(train_clean[num_cols].describe(percentiles=[0.01, 0.5, 0.99]).T)
# Log transforms below reduce skew without learning bounds from validation data.

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
invalid_date_train = X_train.loc[invalid_date_mask(X_train)]
print('Invalid arrival dates after cleaning (train):', len(invalid_date_train))
print('\nInteger-like column types after cleaning:')
print(X_train[[column for column in num_cols if column in X_train.columns]].dtypes)

invalid_date_val = X_val.loc[invalid_date_mask(X_val)]
print('Invalid arrival dates after cleaning (val):', len(invalid_date_val))
print('\nInteger-like column types after cleaning:')
print(X_val[[column for column in num_cols if column in X_val.columns]].dtypes)

invalid_date_test = X_test.loc[invalid_date_mask(X_test)]
print('Invalid arrival dates after cleaning (test):', len(invalid_date_test))
print('\nInteger-like column types after cleaning:')
print(X_test[[column for column in num_cols if column in X_test.columns]].dtypes)

print('--- Cek nilai negatif (seharusnya mustahil untuk SEMUA kolom numerik) ---')
has_negative = False
for column in num_cols:
    n_negative = (X_train[column] < 0).sum()
    n_negative_val = (X_val[column] < 0).sum()
    n_negative_test = (X_test[column] < 0).sum()
    if n_negative > 0 or n_negative_test > 0 or n_negative_val > 0:
        has_negative = True
        print(f'{column}: {n_negative} negatif di train, {n_negative_val} negatif di validation, {n_negative_test} di test — PERLU DITANGANI')
    else:
        print(f'{column}: aman')

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

In [ ]:
class AttributeConstructor(BaseEstimator, TransformerMixin):
    def __init__(self, include_new_features=True):
        # Toggle buat fitur-fitur baru, biar efeknya ke Macro-F1 bisa dicek langsung
        self.include_new_features = include_new_features

    def fit(self, X, y=None):
        # dipelajari cuma dari training rows fold ini, jadi ga ada leakage ke val/test
        self.room_rate_median_ = X.groupby('room_category')['nightly_rate_idr'].median()
        self.global_rate_median_ = X['nightly_rate_idr'].median()
        return self

    def transform(self, X):
        result = X.copy()
        total_nights = result['weekday_nights'] + result['weekend_nights']
        total_guests = result['num_adults'] + result['num_children']
        prior_bookings = result['prior_cancellations'] + result['prior_completed_bookings']
        result['total_guests'] = total_guests
        result['has_children'] = (result['num_children'] > 0).astype(int)
        result['log_total_nights'] = np.log1p(total_nights.clip(lower=0))
        result['weekend_ratio'] = result['weekend_nights'].div(total_nights.replace(0, np.nan)).fillna(0)

        # days_before_arrival ini korelasinya ke target lumayan gede, jadi selain
        # log-transform, worth dicoba juga efek non-linear/threshold-nya
        lead_time = result['days_before_arrival'].clip(lower=0)
        result['log_days_before_arrival'] = np.log1p(lead_time)

        result['log_nightly_rate_idr'] = np.log1p(result['nightly_rate_idr'].clip(lower=0))
        result['rate_per_guest_night'] = result['nightly_rate_idr'].div(total_guests.replace(0, np.nan)).fillna(0)

        if self.include_new_features:
            # booking zero-night (kayak voucher/promo gratis) itu cancel rate-nya
            # jauh lebih kecil dari rata-rata pas di EDA, jadi mending di-flag
            # eksplisit daripada dianggep error data doang
            result['is_zero_night_stay'] = (
                (result['weekend_nights'] == 0) & (result['weekday_nights'] == 0)
            ).astype(int)

            result['days_before_arrival_bucket'] = pd.cut(
                lead_time, bins=[-1, 2, 7, 30, 90, 180, np.inf],
                labels=[0, 1, 2, 3, 4, 5],
            ).astype(int)
            result['is_last_minute'] = (lead_time <= 2).astype(int)
            result['is_far_advance'] = (lead_time >= 180).astype(int)

            # harga mentah ga bisa dibandingin langsung antar room type (Type G
            # emang wajarnya lebih mahal dari Type A), jadi mending dibikin relatif
            # ke median harga room_category yang sama aja 
            room_median = result['room_category'].map(self.room_rate_median_).fillna(self.global_rate_median_)
            result['price_vs_room_median'] = (
                result['nightly_rate_idr'].div(room_median.replace(0, np.nan)).fillna(1.0)
            )

        result['log_total_prior_bookings'] = np.log1p(prior_bookings.clip(lower=0))
        result['cancellation_ratio'] = result['prior_cancellations'].div(prior_bookings.replace(0, np.nan)).fillna(0)
        result['has_booking_history'] = (prior_bookings > 0).astype(int)
        result['month_sin'] = np.sin(2 * np.pi * (result['arrival_month'] - 1) / 12)
        result['month_cos'] = np.cos(2 * np.pi * (result['arrival_month'] - 1) / 12)
        dates = pd.to_datetime(pd.DataFrame({
            'year': result['arrival_year'], 'month': result['arrival_month'],
            'day': result['arrival_day'],
        }, index=result.index), errors='coerce')
        result['invalid_arrival_date'] = dates.isna().astype(int)
        weekday = dates.dt.day_of_week
        result['weekday_sin'] = np.sin(2 * np.pi * weekday / 7).fillna(0)
        result['weekday_cos'] = np.cos(2 * np.pi * weekday / 7).fillna(0)
        return result.drop(columns=[
            'days_before_arrival', 'nightly_rate_idr', 'weekday_nights',
            'weekend_nights', 'num_adults', 'arrival_month', 'arrival_day',
            'prior_cancellations', 'prior_completed_bookings',
        ])

Use compact booking, guest, seasonality, and booking-history features. Replace skewed lead time, price, and stay length with log representations. Keep arrival year once; omit its duplicate year flag and the complement of returning-guest status. Calendar weekday uses actual dates rather than a fixed 30-day cycle. `rate_per_guest_night` divides the nightly booking rate by party size, with zero guests represented as zero. These are hypotheses to validate with macro-F1, not guaranteed improvements; low marginal correlation alone does not justify dropping a feature.

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
# Scale every continuous engineered feature, including price per guest.
to_scale = [
    'num_children', 'total_guests', 'log_total_nights', 'weekend_ratio',
    'log_days_before_arrival', 'log_nightly_rate_idr', 'rate_per_guest_night',
    'arrival_year', 'special_requests', 'log_total_prior_bookings',
    'cancellation_ratio',
]
scale_transformer = ColumnTransformer(
    [('scale', StandardScaler(), to_scale)], remainder='passthrough',
).set_output(transform='pandas')

Continuous features use training-fitted standardization. Binary indicators, one-hot categories, and bounded calendar cycles pass through unchanged.

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
class FeatureEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns=('room_category', 'booking_channel', 'meal_package')):
        self.columns = columns

    def fit(self, X, y=None):
        self.ohe_ = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(X[list(self.columns)])
        return self

    def transform(self, X):
        columns = list(self.columns)
        encoded = pd.DataFrame(
            self.ohe_.transform(X[columns]),
            columns=self.ohe_.get_feature_names_out(columns), index=X.index,
        )
        return pd.concat([X.drop(columns=columns), encoded], axis=1)

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
# Write your code here


## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
class_counts = y_train.value_counts().sort_index()
print(pd.DataFrame({'count': class_counts, 'fraction': class_counts / len(y_train)}))
print('Majority/minority ratio:', class_counts.max() / class_counts.min())
# No resampling is the baseline. Optional oversampling happens only on fit rows.
def resample_training(X, y, method=None):
    if method is None:
        return X, y
    if method != 'random_over':
        raise ValueError("method must be None or 'random_over'")
    rng = np.random.default_rng(CFG.SEED)
    labels = np.asarray(y)
    classes, counts = np.unique(labels, return_counts=True)
    indices = []
    for label, count in zip(classes, counts):
        available = np.flatnonzero(labels == label)
        indices.extend(available)
        indices.extend(rng.choice(available, counts.max() - count, replace=True))
    indices = rng.permutation(indices)
    return X.iloc[indices], y.iloc[indices]

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
# Write your code here

---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
preparation = Pipeline([
    ('clean', DataCleaner()),
    ('construct', AttributeConstructor()),
    ('encode', FeatureEncoder()),
    ('scale', scale_transformer),
])
# Preview only. Evaluation below fits a fresh clone on each fold's raw fit rows.
prepared = clone(preparation).fit(X_train_raw, y_train)
X_train_prepared = prepared.transform(X_train_raw)
X_val_prepared = prepared.transform(X_val_raw)
X_test_prepared = prepared.transform(X_test)
X_train, X_val = X_train_raw.copy(), X_val_raw.copy()
assert CFG.TARGET not in X_test_prepared.columns
assert np.isfinite(X_train_prepared.to_numpy(dtype=float)).all()
print('Prepared shapes:', X_train_prepared.shape, X_val_prepared.shape, X_test_prepared.shape)
print('Target correlations (training only, diagnostic):')
print(X_train_prepared.assign(is_canceled=y_train).corr()['is_canceled'].drop('is_canceled').sort_values())

---

# Modeling and Validation <a name="7"></a>

---

In [ ]:
def fit_model(model, X_fit, y_fit, method=None):
    prep = clone(preparation)
    transformed = prep.fit_transform(X_fit, y_fit)
    X_resampled, y_resampled = resample_training(transformed, y_fit, method)
    fitted_model = clone(model).fit(X_resampled, y_resampled)
    return Pipeline([('preparation', prep), ('model', fitted_model)])

def fit_predict(model, X_fit, y_fit, X_eval, method=None):
    return fit_model(model, X_fit, y_fit, method).predict(X_eval)

def holdout(models, X_train, y_train, X_val, y_val, method=None):
    scores = {}

    for name, model in models.items():
        predicted = fit_predict(model, X_train, y_train, X_val, method)
        scores[name] = f1_score(y_val, predicted, average='macro')

    return pd.Series(scores, name='holdout').to_frame()

In [ ]:
def kfold_scores(model, X, y, method=None):
    folds = StratifiedKFold(n_splits=Settings.N_FOLDS, shuffle=True, random_state=Settings.SEED)
    scores = []

    for fit_index, eval_index in folds.split(X, y):
        predicted = fit_predict(model, X.iloc[fit_index], y.iloc[fit_index], X.iloc[eval_index], method)
        scores.append(f1_score(y.iloc[eval_index], predicted, average='macro'))

    return np.array(scores)

def kfold(models, X, y, method=None):
    rows = {}

    for name, model in models.items():
        scores = kfold_scores(model, X, y, method)
        rows[name] = {'kfold_mean': scores.mean(), 'kfold_std': scores.std()}

    return pd.DataFrame(rows).T

In [ ]:
def compare(holdout_table, kfold_table):
    return holdout_table.join(kfold_table).sort_values('kfold_mean', ascending=False)

In [ ]:
def tune(model, grid, X, y, method=None):
    rows = []

    for params in ParameterGrid(grid):
        scores = kfold_scores(clone(model).set_params(**params), X, y, method)
        rows.append({**params, 'kfold_mean': scores.mean(), 'kfold_std': scores.std()})

    return pd.DataFrame(rows).sort_values('kfold_mean', ascending=False, ignore_index=True)

In [ ]:
from ml_lib.model import KNN
from ml_lib.model.dtl import DecisionTreeCART

## KNN


In [ ]:
knn = KNN(k=5, p=2)
knn

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
dtl = DecisionTreeCART(min_samples_split=2, max_depth=10)
dtl

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
# lr = LogisticRegression(learning_rate=0.01, n_iters=1000)
# lr

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
# Write your code here

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
# Write your code here

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

This part compares the validation results of all models.

In [ ]:
models = {'knn': knn, 
          'dtl': dtl, 
        #   'svc': svc, 
          # 'lr': lr, 
        #   'gnb': gnb
          }
models

In [ ]:
holdout_table = holdout(models, X_train, y_train, X_val, y_val)
print('Macro-F1 comparison; validation retains its original class distribution:')
balance_comparison = holdout_table.rename(columns={'holdout': 'no_resampling'}).join(
    holdout(models, X_train, y_train, X_val, y_val, method='random_over')
    .rename(columns={'holdout': 'random_oversampling'})
)
balance_comparison

In [ ]:
kfold_table = kfold(models, X_train, y_train)
kfold_table

In [ ]:
compare(holdout_table, kfold_table)

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
gridKNN = {
    'k': [3,5,7,9,23,57,99], 
    'p': [1,2,3], 
}


In [ ]:
gridDTL = {
    'min_samples_split': [2, 10, 30],
    'max_depth': [5, 10, 20],
    'ccp_alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
}

In [ ]:
gridLR = {
    'learning_rate': [0.1, 0.5, 1],
    'n_iters': [50, 100, 300, 1000],
}


In [ ]:
knn_tuned = tune(models['knn'], gridKNN, X_train, y_train)
dtl_tuned = tune(models['dtl'], gridDTL, X_train, y_train)

In [ ]:
tuned_models = {'knn': knn_tuned, 
                'dtl': dtl_tuned, 
                # 'svc': svc_tuned, 
                # 'gnb': gnb_tuned
                }
tuned_models

In [ ]:
tuning_results = {
    'knn': knn_tuned,
    'dtl': dtl_tuned,
    # 'svc': svc_tuned,
    # 'gnb': gnb_tuned,
}

metric_columns = ['kfold_mean', 'kfold_std']
tuned_models = {
    name: clone(models[name]).set_params(
        **result.drop(columns=metric_columns).to_dict(orient='records')[0]
    )
    for name, result in tuning_results.items()
}

kfold_table_tuned = kfold(tuned_models, X_train, y_train)
kfold_table_tuned

holdout_table_tuned = holdout(tuned_models, X_train, y_train, X_val, y_val)
tuned_comparison = compare(holdout_table_tuned, kfold_table_tuned)

best_model_name = tuned_comparison.index[0]
tuned_comparison

---

# Comparison with Scikit Learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## KNN

Model K-Nearest Neighbors pada variabel `knn`.

<!-- Use n_neighbors 17 -->

In [ ]:
knn_scikit = KNeighborsClassifier(5, metric="minkowski")
knn_scikit

## DTL

Model Decision Tree pada variabel `dtl`.

In [ ]:
dtl_scikit = DecisionTreeClassifier(max_depth=10, min_samples_split=2, min_samples_leaf=1)
dtl_scikit

### Cost-Complexity Pruning Path

This part inspects how `ccp_alpha` trades tree size for Macro-F1, to justify the pruning hyperparameter search above instead of guessing a range blindly.

In [ ]:
# --- Cost-Complexity Pruning Path (model optimization technique, bonus) ---
prep_for_path = clone(preparation).fit(X_train, y_train)
X_train_path = prep_for_path.transform(X_train)
X_val_path = prep_for_path.transform(X_val)

path = dtl_scikit.cost_complexity_pruning_path(X_train_path, y_train)
ccp_alphas = path.ccp_alphas[:-1]  # drop the trivial last alpha (root-only tree)

trees = [
    clone(dtl_scikit).set_params(ccp_alpha=alpha).fit(X_train_path, y_train)
    for alpha in ccp_alphas
]
node_counts = [tree.tree_.node_count for tree in trees]
train_f1 = [f1_score(y_train, tree.predict(X_train_path), average='macro') for tree in trees]
val_f1 = [f1_score(y_val, tree.predict(X_val_path), average='macro') for tree in trees]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ccp_alphas, node_counts, marker='o', drawstyle='steps-post')
axes[0].set_xlabel('ccp_alpha')
axes[0].set_ylabel('Number of nodes')
axes[0].set_title('Tree size vs ccp_alpha')

axes[1].plot(ccp_alphas, train_f1, marker='o', label='Train Macro-F1', drawstyle='steps-post')
axes[1].plot(ccp_alphas, val_f1, marker='o', label='Validation Macro-F1', drawstyle='steps-post')
axes[1].set_xlabel('ccp_alpha')
axes[1].set_ylabel('Macro-F1')
axes[1].set_title('Macro-F1 vs ccp_alpha')
axes[1].legend()

plt.tight_layout()
plt.show()

best_idx = int(np.argmax(val_f1))
print(f'Best ccp_alpha by validation Macro-F1: {ccp_alphas[best_idx]:.6f} '
      f'(val F1={val_f1[best_idx]:.4f}, nodes={node_counts[best_idx]})')

## SVM

Model Support Vector Machine pada variabel `svc`.

In [ ]:
svc_scikit = SVC(kernel="rbf")
svc_scikit

## Logistic Regression

Model Logistic Regression pada variabel `lr`.

In [ ]:
lr_scikit = SklearnLogisticRegression(solver='liblinear', max_iter=150)
lr_scikit

## Gaussian NB

Model Gaussian Naive Bayes pada variabel `gnb`.

In [ ]:
gnb_scikit = GaussianNB(var_smoothing=1e-8)
gnb_scikit

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

This part compares the validation results of all models.

In [ ]:
models_scikit = {'knn': knn_scikit, 
                 'dtl': dtl_scikit, 
                 'svc': svc_scikit, 
                 'lr': lr_scikit, 
                 'gnb': gnb_scikit}
models_scikit

In [ ]:
for model in models_scikit : 
    print (model)
    fit_predict (models_scikit[model], X_train, y_train, X_val)

holdout_table_scikit = holdout(models_scikit, X_train, y_train, X_val, y_val, method=None)   
holdout_table_scikit

In [ ]:
kfold_table_scikit = kfold(models_scikit, X_train, y_train)
kfold_table_scikit

In [ ]:
compare(holdout_table_scikit, kfold_table_scikit)

## Mutual Information Check

Pearson correlation only captures linear relationships, so it understates threshold/non-linear features such as `is_zero_night_stay` or `is_last_minute`. Mutual information captures non-linear dependence as well, and is checked here per feature as a complement to the ablation test below (which checks the effect on Macro-F1 per model).

In [ ]:
mi_scores = mutual_info_classif(
    X_train_prepared, y_train, discrete_features='auto', random_state=CFG.SEED,
)
mi_table = pd.Series(mi_scores, index=X_train_prepared.columns, name='mutual_info').sort_values(ascending=False)
mi_table

## Feature Engineering Ablation

The new engineered features (`is_zero_night_stay`, `days_before_arrival_bucket`, `is_last_minute`, `is_far_advance`, `price_vs_room_median`) have near-zero Pearson correlation with the target. Pearson correlation only captures linear relationships, so a low score does not necessarily mean the feature is noise -- it is checked here directly against Macro-F1 instead.

In [ ]:
preparation_with = Pipeline([
    ('clean', DataCleaner()),
    ('construct', AttributeConstructor(include_new_features=True)),
    ('encode', FeatureEncoder()),
    ('scale', scale_transformer),
])
preparation_without = Pipeline([
    ('clean', DataCleaner()),
    ('construct', AttributeConstructor(include_new_features=False)),
    ('encode', FeatureEncoder()),
    ('scale', scale_transformer),
])

def kfold_with_prep(prep, model, X, y):
    folds = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    scores = []
    for fit_idx, eval_idx in folds.split(X, y):
        pipeline = Pipeline([('preparation', clone(prep)), ('model', clone(model))])
        pipeline.fit(X.iloc[fit_idx], y.iloc[fit_idx])
        pred = pipeline.predict(X.iloc[eval_idx])
        scores.append(f1_score(y.iloc[eval_idx], pred, average='macro'))
    return np.array(scores)

ablation_rows = []
for name, model in models_scikit.items():
    with_score = kfold_with_prep(preparation_with, model, X_train, y_train).mean()
    without_score = kfold_with_prep(preparation_without, model, X_train, y_train).mean()
    ablation_rows.append({
        'model': name, 'with_new_features': with_score,
        'without_new_features': without_score, 'diff': with_score - without_score,
    })

ablation_table = pd.DataFrame(ablation_rows).set_index('model')
ablation_table

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
gridKNN_scikit = {
    'n_neighbors': [5], 
    'metric': ['minkowski'], 
    'weights': ['uniform', 'distance']
}


In [ ]:
gridDTL_scikit = {
    'max_depth': [10],
    'min_samples_split': [2],
    'min_samples_leaf': [1],
    'ccp_alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
}

In [ ]:
gridSVM_scikit ={
    'kernel' : ['rbf'],
    'C' : [0.1],
    'gamma' : ['scale', 0.01, 0.1]
}


In [ ]:
gridLR_scikit = {
    'max_iter': [50],
    'penalty': ['l2'],
    'C': [0.1]
}


In [ ]:
gridGNB_scikit ={
    'var_smoothing': [1e-9]
}


In [ ]:
knn_tuned_scikit = tune (models_scikit['knn'], gridKNN_scikit, X_train, y_train)
dtl_tuned_scikit = tune(models_scikit['dtl'], gridDTL_scikit, X_train, y_train)
svc_tuned_scikit = tune(models_scikit['svc'], gridSVM_scikit, X_train, y_train)
lr_tuned_scikit = tune(models_scikit['lr'], gridLR_scikit, X_train, y_train)
gnb_tuned_scikit = tune(models_scikit['gnb'], gridGNB_scikit, X_train, y_train)

In [ ]:
tuned_models = {'knn': knn_tuned_scikit, 
                'dtl': dtl_tuned_scikit, 
                'svc': svc_tuned_scikit, 
                'lr': lr_tuned_scikit, 
                'gnb': gnb_tuned_scikit}
tuned_models

In [ ]:
tuning_results_scikit = {
    'knn': knn_tuned_scikit,
    'dtl': dtl_tuned_scikit,
    'svc': svc_tuned_scikit,
    'lr': lr_tuned_scikit,
    'gnb': gnb_tuned_scikit,
}

metric_columns = ['kfold_mean', 'kfold_std']
tuned_models_scikit = {
    name: clone(models_scikit[name]).set_params(
        **result.drop(columns=metric_columns).to_dict(orient='records')[0]
    )
    for name, result in tuning_results_scikit.items()
}

kfold_table_tuned_scikit = kfold(tuned_models_scikit, X_train, y_train)
kfold_table_tuned_scikit

holdout_table_tuned_scikit = holdout(tuned_models_scikit, X_train, y_train, X_val, y_val)
tuned_comparison_scikit = compare(holdout_table_tuned_scikit, kfold_table_tuned_scikit)

best_model_name_scikit = tuned_comparison_scikit.index[0]
tuned_comparison_scikit

In [ ]:
# Write your code here
tuned_comparison
tuned_comparison_scikit

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [ ]:
# Write your code here

def save_model(model, name):
    path = CFG.OUTPUT_DIR / f'{name}.pkl'
    joblib.dump(model, path)
    return path

def load_model(name):
    return joblib.load(CFG.OUTPUT_DIR / f'{name}.pkl')

In [ ]:
for name, model in models.items():
    fitted = fit_model(model, X_train, y_train)
    print(f'{name}: saved to {save_model(fitted, name)}')

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
def submit(model, X, y, X_test, method=None):
    predictions = fit_predict(model, X, y, X_test, method)
    ids = test_df.loc[X_test.index, CFG.ID_COLUMN]
    result = pd.DataFrame({CFG.ID_COLUMN: ids, CFG.TARGET: predictions})
    assert len(result) == len(test_df)
    path = CFG.OUTPUT_DIR / 'submission.csv'
    result.to_csv(path, index=False)
    print(f'{len(result)} rows written to {path}')
    return result.head()

In [ ]:
X_final = pd.concat([X_train, X_val])
y_final = pd.concat([y_train, y_val])

submit(tuned_models[best_model_name], X_final, y_final, X_test)

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.

### Confusion Matrix

Confusion matrix serta precision dan recall kelima model.

In [ ]:
def confusion(models, X_train, y_train, X_val, y_val, method=None):
    figure, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 3.6), squeeze=False)

    for axis, (name, model) in zip(axes.flat, models.items()):
        predicted = fit_predict(model, X_train, y_train, X_val, method)
        ConfusionMatrixDisplay.from_predictions(y_val, predicted, ax=axis, colorbar=False)
        axis.set_title(name)

    plt.tight_layout()
    plt.show()

def report(models, X_train, y_train, X_val, y_val, method=None):
    tables = {}

    for name, model in models.items():
        predicted = fit_predict(model, X_train, y_train, X_val, method)
        tables[name] = pd.DataFrame(classification_report(y_val, predicted, output_dict=True)).T

    return pd.concat(tables)

In [ ]:
confusion (tuned_models, X_train, y_train, X_val, y_val)
confusion (tuned_models_scikit, X_train, y_train, X_val, y_val)

In [ ]:
report (tuned_models, X_train, y_train, X_val, y_val)
report (tuned_models_scikit, X_train, y_train, X_val, y_val)

### Misclassified Patterns

Baris asli `train` yang salah diprediksi oleh model terbaik.

In [ ]:
def misclassified(model, X_train, y_train, X_val, y_val, method=None):
    predicted = pd.Series(fit_predict(model, X_train, y_train, X_val, method), index=X_val.index)
    wrong = predicted[predicted != y_val]
    return train_df.loc[wrong.index].assign(predicted=wrong)

In [ ]:
misclassified(tuned_models[best_model_name], X_train, y_train, X_val, y_val)
misclassified(tuned_models_scikit[best_model_name_scikit], X_train, y_train, X_val, y_val)